# Эксперимент 2 — Метрики Entity Resolution (без Schema Matching)

Ноутбук для **валидации пайплайна** на синтетическом датасете и
**визуализации метрик** после запуска `02_train_er_without_sm.py`.

Структура:
1. Импорты и конфигурация
2. Загрузка синтетических данных и предвычисленных эмбеддингов
3. Валидация целостности данных
4. Построение графа и обучение (один датасет)
5. Метрики на тестовой выборке
6. Визуализация метрик (bar chart, PR-curve, ROC-curve)
7. Анализ дубликатов
8. Кривые обучения (train/val loss)

## 1. Импорты и конфигурация

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    auc,
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
)
from IPython.display import display

# ── Paths ──────────────────────────────────────────────────────────
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

DATA_DIR   = ROOT / "data"
SYNTH_DIR  = DATA_DIR / "synthetic"
EMB_DIR    = DATA_DIR / "embeddings"
OUTPUT_DIR = ROOT / "output"

# ── Project imports ────────────────────────────────────────────────
from table_unifier.config import Config, EntityResolutionConfig
from table_unifier.dataset.embedding_generation import TokenEmbedder
from table_unifier.dataset.graph_builder import build_graph
from table_unifier.dataset.pair_sampling import (
    build_triplet_indices, mine_hard_negatives, split_labeled_pairs,
)
from table_unifier.training.er_trainer import (
    find_duplicates, get_row_embeddings, train_entity_resolution,
)

# ── Config ─────────────────────────────────────────────────────────
config = Config(data_dir=DATA_DIR, output_dir=OUTPUT_DIR)
er_cfg  = config.entity_resolution
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Dataset to use for single-dataset sections
DATASET_NAME = "beer"  # ← change to any available dataset

In [ ]:
# ── Helper functions (inlined from 02_train_er_without_sm.py) ─────

def load_synth_dataset(name: str) -> dict | None:
    """Загрузить синтетический датасет и предвычисленные эмбеддинги."""
    ds_synth = SYNTH_DIR / name
    ds_emb   = EMB_DIR   / name

    required = [
        ds_synth / "tableA_synth.csv", ds_synth / "tableB_synth.csv",
        ds_synth / "train.csv",
        ds_emb / "column_embeddings_a.npz", ds_emb / "column_embeddings_b.npz",
        ds_emb / "row_embeddings_a.npy",    ds_emb / "row_embeddings_b.npy",
    ]
    missing = [p for p in required if not p.exists()]
    if missing:
        print(f"[{name}] Отсутствуют: {[p.name for p in missing]}")
        return None

    splits = {}
    for split in ("train", "valid", "test"):
        p = ds_synth / f"{split}.csv"
        if p.exists():
            splits[split] = pd.read_csv(p)

    return {
        "table_a":   pd.read_csv(ds_synth / "tableA_synth.csv"),
        "table_b":   pd.read_csv(ds_synth / "tableB_synth.csv"),
        "splits":    splits,
        "col_emb_a": dict(np.load(ds_emb / "column_embeddings_a.npz")),
        "col_emb_b": dict(np.load(ds_emb / "column_embeddings_b.npz")),
        "row_emb_a": np.load(ds_emb / "row_embeddings_a.npy"),
        "row_emb_b": np.load(ds_emb / "row_embeddings_b.npy"),
    }


def evaluate_test(embeddings, test_df, id_to_global_a, id_to_global_b):
    """Вернуть метрики + массивы scores/labels для построения кривых."""
    positives, negatives = split_labeled_pairs(test_df)
    scores, labels = [], []
    for a_id, b_id in positives:
        ga, gb = id_to_global_a.get(a_id), id_to_global_b.get(b_id)
        if ga is not None and gb is not None:
            scores.append((embeddings[ga] @ embeddings[gb]).item())
            labels.append(1)
    for a_id, b_id in negatives:
        ga, gb = id_to_global_a.get(a_id), id_to_global_b.get(b_id)
        if ga is not None and gb is not None:
            scores.append((embeddings[ga] @ embeddings[gb]).item())
            labels.append(0)
    if not labels:
        return {}
    s = np.array(scores); l = np.array(labels)
    best_f1, best_thr = 0.0, 0.5
    for thr in np.linspace(s.min(), s.max(), 100):
        f1 = f1_score(l, (s >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    preds = (s >= best_thr).astype(int)
    return {
        "threshold":     best_thr,
        "precision":     float(precision_score(l, preds, zero_division=0)),
        "recall":        float(recall_score(l, preds, zero_division=0)),
        "f1":            float(best_f1),
        "roc_auc":       float(roc_auc_score(l, s)),
        "avg_precision": float(average_precision_score(l, s)),
        "n_pos": int(l.sum()), "n_neg": int((1 - l).sum()),
        "_scores": s, "_labels": l,
    }

## 2. Загрузка синтетического датасета и предвычисленных эмбеддингов

In [ ]:
ds = load_synth_dataset(DATASET_NAME)
assert ds is not None, f"Датасет '{DATASET_NAME}' не найден"

table_a = ds["table_a"]
table_b = ds["table_b"]
row_emb_a, row_emb_b = ds["row_emb_a"], ds["row_emb_b"]
col_emb_a, col_emb_b = ds["col_emb_a"], ds["col_emb_b"]

print(f"Table A: {table_a.shape}  |  row_emb_a: {row_emb_a.shape}")
print(f"Table B: {table_b.shape}  |  row_emb_b: {row_emb_b.shape}")
print(f"Col emb A keys ({len(col_emb_a)}): {list(col_emb_a.keys())}")
print(f"Col emb B keys ({len(col_emb_b)}): {list(col_emb_b.keys())}")
print(f"\nSplits available: {list(ds['splits'].keys())}")
for split, df in ds["splits"].items():
    pos = (df["label"] == 1).sum()
    print(f"  {split:6s}: {len(df):4d} rows  pos={pos}  neg={len(df)-pos}")

print("\nTable A (first 3 rows):")
display(table_a.head(3))
print("Table B (first 3 rows):")
display(table_b.head(3))

## 3. Валидация целостности данных

In [ ]:
checks = []

def chk(name, ok, detail=""):
    status = "✅ PASS" if ok else "❌ FAIL"
    checks.append((name, status, detail))
    print(f"{status}  {name}" + (f"  ({detail})" if detail else ""))

# Row count matches embedding shape
chk("row_emb_a rows == len(table_a)",
    row_emb_a.shape[0] == len(table_a),
    f"{row_emb_a.shape[0]} vs {len(table_a)}")
chk("row_emb_b rows == len(table_b)",
    row_emb_b.shape[0] == len(table_b),
    f"{row_emb_b.shape[0]} vs {len(table_b)}")

# No NaN/Inf in row embeddings
chk("No NaN in row_emb_a", not np.isnan(row_emb_a).any())
chk("No NaN in row_emb_b", not np.isnan(row_emb_b).any())
chk("No Inf in row_emb_a", not np.isinf(row_emb_a).any())
chk("No Inf in row_emb_b", not np.isinf(row_emb_b).any())

# Column embedding keys match table columns
cols_a = [c for c in table_a.columns if c != "id"]
cols_b = [c for c in table_b.columns if c != "id"]
missing_a = set(cols_a) - set(col_emb_a.keys())
missing_b = set(cols_b) - set(col_emb_b.keys())
chk("All table_a cols have col embeddings", len(missing_a) == 0,
    f"missing: {missing_a}" if missing_a else "")
chk("All table_b cols have col embeddings", len(missing_b) == 0,
    f"missing: {missing_b}" if missing_b else "")

# Split IDs exist in table A/B
ids_a = set(table_a["id"].astype(str))
ids_b = set(table_b["id"].astype(str))
for split, df in ds["splits"].items():
    bad_a = set(df["ltable_id"].astype(str)) - ids_a
    bad_b = set(df["rtable_id"].astype(str)) - ids_b
    chk(f"{split}: ltable_ids in table_a", len(bad_a) == 0,
        f"{len(bad_a)} unknown IDs" if bad_a else "")
    chk(f"{split}: rtable_ids in table_b", len(bad_b) == 0,
        f"{len(bad_b)} unknown IDs" if bad_b else "")

print(f"\n{'='*40}")
n_ok = sum(1 for _, s, _ in checks if "PASS" in s)
print(f"Result: {n_ok}/{len(checks)} checks passed")

## 4. Построение графа и обучение

In [ ]:
token_embedder = TokenEmbedder(model_name=er_cfg.token_model_name, device=DEVICE)

column_embeddings = {**col_emb_a, **col_emb_b}
er_cfg.row_dim  = row_emb_a.shape[1]
er_cfg.token_dim = token_embedder.hidden_dim
er_cfg.col_dim  = next(iter(column_embeddings.values())).shape[0]

id_to_idx_a = {str(r["id"]): i for i, (_, r) in enumerate(table_a.iterrows())}
id_to_idx_b = {str(r["id"]): i for i, (_, r) in enumerate(table_b.iterrows())}

# Hard-negative triplets for train and validation
positives_train, _ = split_labeled_pairs(ds["splits"]["train"])
hard_train = mine_hard_negatives(row_emb_a, row_emb_b, positives_train,
                                  id_to_idx_a, id_to_idx_b, top_k=5)

graph, id_to_global_a, id_to_global_b = build_graph(
    table_a, table_b, column_embeddings, token_embedder,
    columns_a=cols_a, columns_b=cols_b,
    precomputed_row_embeddings_a=row_emb_a,
    precomputed_row_embeddings_b=row_emb_b,
)
print(f"Graph: row={graph['row'].x.shape}, token={graph['token'].x.shape}")

train_idx = build_triplet_indices(hard_train, id_to_global_a, id_to_global_b)
print(f"Train triplets: {len(train_idx)}")

val_idx = None
if "valid" in ds["splits"]:
    positives_val, _ = split_labeled_pairs(ds["splits"]["valid"])
    hard_val = mine_hard_negatives(row_emb_a, row_emb_b, positives_val,
                                    id_to_idx_a, id_to_idx_b, top_k=3)
    if hard_val:
        val_idx = build_triplet_indices(hard_val, id_to_global_a, id_to_global_b)
        print(f"Val triplets:   {len(val_idx)}")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_path = OUTPUT_DIR / f"er_model_{DATASET_NAME}.pt"

er_cfg.epochs = 30  # быстрее для ноутбука; увеличьте для полного обучения
model, history = train_entity_resolution(
    data=graph,
    triplet_indices=train_idx,
    config=er_cfg,
    val_triplet_indices=val_idx,
    device=DEVICE,
    save_path=save_path,
)
embeddings = get_row_embeddings(model, graph, device=DEVICE)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Model saved: {save_path.exists()}")

## 5. Метрики на тестовой выборке

In [ ]:
test_metrics = evaluate_test(embeddings, ds["splits"]["test"], id_to_global_a, id_to_global_b)

# ── Итоговая таблица метрик ────────────────────────────────────────
df_metrics = pd.DataFrame([{
    "dataset":       DATASET_NAME,
    "F1":            test_metrics["f1"],
    "Precision":     test_metrics["precision"],
    "Recall":        test_metrics["recall"],
    "ROC-AUC":       test_metrics["roc_auc"],
    "Avg Precision": test_metrics["avg_precision"],
    "threshold":     test_metrics["threshold"],
    "n_pos":         test_metrics["n_pos"],
    "n_neg":         test_metrics["n_neg"],
}])

display(
    df_metrics.style
    .format({c: "{:.4f}" for c in ["F1","Precision","Recall","ROC-AUC","Avg Precision","threshold"]})
    .highlight_max(subset=["F1","Precision","Recall","ROC-AUC"], color="lightgreen")
)

# ── Если есть er_results.json, показать все датасеты ─────────────
results_path = OUTPUT_DIR / "er_results.json"
if results_path.exists():
    with open(results_path, encoding="utf-8") as f:
        er_results = json.load(f)
    all_rows = er_results.get("results", [])
    if all_rows:
        df_all = pd.DataFrame(all_rows)
        numeric_cols = ["f1", "precision", "recall", "roc_auc", "avg_precision"]
        numeric_cols = [c for c in numeric_cols if c in df_all.columns]
        print("\n── Все датасеты (из er_results.json):")
        display(
            df_all[["name", "domain"] + numeric_cols].style
            .format({c: "{:.4f}" for c in numeric_cols})
            .highlight_max(subset=numeric_cols, color="lightgreen")
            .background_gradient(subset=numeric_cols, cmap="YlGn")
        )

## 6. Визуализация метрик

In [ ]:
scores  = test_metrics["_scores"]
labels  = test_metrics["_labels"]
thr_opt = test_metrics["threshold"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── (A) PR-curve ──────────────────────────────────────────────────
prec, rec, _ = precision_recall_curve(labels, scores)
ap = test_metrics["avg_precision"]
axes[0].plot(rec, prec, lw=2, color="steelblue", label=f"AP={ap:.3f}")
axes[0].axhline(labels.mean(), ls="--", color="grey", lw=1, label="baseline")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve"); axes[0].legend(); axes[0].grid(alpha=0.3)

# ── (B) ROC-curve ─────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(labels, scores)
roc = test_metrics["roc_auc"]
axes[1].plot(fpr, tpr, lw=2, color="coral", label=f"AUC={roc:.3f}")
axes[1].plot([0,1],[0,1], ls="--", color="grey", lw=1)
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC Curve"); axes[1].legend(); axes[1].grid(alpha=0.3)

# ── (C) Score distribution + threshold ────────────────────────────
bins = 50
axes[2].hist(scores[labels == 0], bins=bins, alpha=0.6, color="salmon",  label="negative")
axes[2].hist(scores[labels == 1], bins=bins, alpha=0.6, color="seagreen", label="positive")
axes[2].axvline(thr_opt, color="black", lw=2, ls="--", label=f"threshold={thr_opt:.3f}")
axes[2].set_xlabel("Cosine similarity"); axes[2].set_ylabel("Count")
axes[2].set_title("Score Distribution"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f"{DATASET_NAME} — Test Metrics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart of F1/P/R/AUC across all datasets (from er_results.json) ──
if results_path.exists():
    df_bar = pd.DataFrame(er_results["results"])
    metric_cols = [c for c in ["f1", "precision", "recall", "roc_auc"] if c in df_bar.columns]
    df_melt = df_bar[["name"] + metric_cols].melt(id_vars="name", var_name="metric", value_name="value")

    fig, ax = plt.subplots(figsize=(max(12, len(df_bar) * 0.7), 5))
    sns.barplot(data=df_melt, x="name", y="value", hue="metric", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score"); ax.set_xlabel("")
    ax.set_title("ER без Schema Matching — метрики по всем датасетам")
    ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("er_results.json не найден — запустите 02_train_er_without_sm.py --all")

## 7. Анализ дубликатов

In [ ]:
DUP_THRESHOLD = 0.7
duplicates = find_duplicates(embeddings, id_to_global_a, id_to_global_b, threshold=DUP_THRESHOLD)
print(f"Найдено дубликатов (sim >= {DUP_THRESHOLD}): {len(duplicates)}")

# Показать первые 20 пар с их сходством
ids_a_inv = {v: k for k, v in id_to_global_a.items()}
ids_b_inv = {v: k for k, v in id_to_global_b.items()}

dup_rows = []
for ga, gb in duplicates[:20]:
    sim = (embeddings[ga] @ embeddings[gb]).item()
    dup_rows.append({
        "id_a": ids_a_inv.get(ga, ga),
        "id_b": ids_b_inv.get(gb, gb),
        "cosine_sim": round(sim, 4),
    })
if dup_rows:
    df_dups = pd.DataFrame(dup_rows)
    display(df_dups.sort_values("cosine_sim", ascending=False))

In [ ]:
# ── Heatmap: similarity sample (first 30 A × 30 B rows) ──────────
sample_a = list(id_to_global_a.values())[:30]
sample_b = list(id_to_global_b.values())[:30]
sim_matrix = (embeddings[sample_a] @ embeddings[sample_b].T).numpy()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1, ax=ax,
            xticklabels=False, yticklabels=False)
ax.set_title(f"{DATASET_NAME}: Cosine similarity (sample 30×30 A×B)")
ax.set_xlabel("Table B rows"); ax.set_ylabel("Table A rows")
plt.tight_layout()
plt.show()

## 8. Кривые обучения (train / val loss)

In [ ]:
def plot_history(histories: dict[str, dict], ax=None):
    """Нарисовать train/val loss.
    histories: {dataset_name: {"train_loss": [...], "val_loss": [...]}}
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    colors = plt.cm.tab10.colors

    for i, (name, h) in enumerate(histories.items()):
        color = colors[i % len(colors)]
        epochs = range(1, len(h["train_loss"]) + 1)
        ax.plot(epochs, h["train_loss"], lw=2, color=color, label=f"{name} train")
        if h.get("val_loss"):
            ax.plot(epochs, h["val_loss"], lw=1.5, ls="--", color=color,
                    alpha=0.7, label=f"{name} val")
            best_ep = int(np.argmin(h["val_loss"])) + 1
            best_v  = min(h["val_loss"])
            ax.axvline(best_ep, color=color, lw=0.8, ls=":", alpha=0.5)
            ax.scatter([best_ep], [best_v], color=color, s=60, zorder=5)

    ax.set_xlabel("Epoch"); ax.set_ylabel("Triplet Loss")
    ax.set_title("Training history"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    return ax


# ── Single dataset from current training run ────────────────────
plot_history({DATASET_NAME: history})
plt.tight_layout(); plt.show()

# ── All datasets from saved .history.json files ──────────────────
all_histories = {}
for p in OUTPUT_DIR.glob("er_model_*.history.json"):
    ds_name = p.stem.replace("er_model_", "").replace(".history", "")
    with open(p, encoding="utf-8") as f:
        all_histories[ds_name] = json.load(f)

if len(all_histories) > 1:
    fig, ax = plt.subplots(figsize=(14, 6))
    plot_history(all_histories, ax=ax)
    plt.tight_layout(); plt.show()
    print(f"Показаны кривые для {len(all_histories)} датасетов")